In [5]:
"""
Classical Shor's Algorithm - Essential Metrics Only
Clean data with proper timing and fixed algorithm
"""

import math
import numpy as np
import time

# ============================================================================
# CORE FUNCTIONS
# ============================================================================

def gcd(a, b):
    """Compute GCD."""
    while b:
        a, b = b, a % b
    return a

def classical_order(x, N):
    """Find order r where x^r ≡ 1 (mod N)."""
    r = 1
    value = x % N
    operations = 0
    while value != 1:
        value = (value * x) % N
        r += 1
        operations += 1
        if r > N:  # Safety check
            return None, operations
    return r, operations

def is_prime(n):
    """Check if n is prime."""
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(math.sqrt(n)) + 1, 2):
        if n % i == 0:
            return False
    return True

def factorize(N):
    """Get the prime factorization of N."""
    factors = []
    d = 2
    temp_N = N
    while d * d <= temp_N:
        while temp_N % d == 0:
            factors.append(d)
            temp_N //= d
        d += 1
    if temp_N > 1:
        factors.append(temp_N)
    return factors

def analyze_single_base(N, a):
    """Analyze a single base."""
    result = {
        'success': False,
        'operations': 0,
        'failure_reason': None
    }
    
    # Check if coprime
    g = gcd(a, N)
    result['operations'] += 1
    
    if g != 1:
        result['success'] = True
        result['method'] = 'gcd'
        return result
    
    # Find order
    r, ops = classical_order(a, N)
    result['operations'] += ops
    
    if r is None:
        result['failure_reason'] = 'order_not_found'
        return result
    
    if r % 2 != 0:
        result['failure_reason'] = 'odd_period'
        return result
    
    # Check x^(r/2) ≡ -1 (mod N)
    x_r_half = pow(a, r // 2, N)
    result['operations'] += 1
    
    if x_r_half == N - 1:
        result['failure_reason'] = 'x^(r/2)=-1'
        return result
    
    # Extract factors
    p = gcd(x_r_half - 1, N)
    q = gcd(x_r_half + 1, N)
    result['operations'] += 2
    
    # Check if we found non-trivial factors
    if (p > 1 and p < N) or (q > 1 and q < N):
        result['success'] = True
        result['method'] = 'period_finding'
    else:
        result['failure_reason'] = 'trivial_factors'
    
    return result

def analyze_N(N):
    """Analyze all bases for a given N - returns essential metrics only."""
    valid_bases = [a for a in range(2, N) if gcd(a, N) == 1]
    
    success_count = 0
    operations_list = []
    
    # Warm-up run to exclude initialization overhead
    _ = analyze_single_base(N, valid_bases[0])
    
    # Actual timed run
    start_time = time.perf_counter()
    
    for a in valid_bases:
        result = analyze_single_base(N, a)
        operations_list.append(result['operations'])
        if result['success']:
            success_count += 1
    
    total_time = (time.perf_counter() - start_time) * 1000  # Convert to ms
    
    return {
        'N': N,
        'factors': factorize(N),
        'num_bases': len(valid_bases),
        'success_count': success_count,
        'failure_count': len(valid_bases) - success_count,
        'success_rate': (success_count / len(valid_bases) * 100) if valid_bases else 0,
        'avg_operations': np.mean(operations_list),
        'time_ms': total_time
    }

# ============================================================================
# MAIN ANALYSIS
# ============================================================================

def main():
    """Run analysis and print essential metrics only."""
    
    print("\n" + "="*80)
    print(" "*20 + "CLASSICAL SHOR'S ALGORITHM")
    print(" "*25 + "ESSENTIAL METRICS")
    print("="*80)
    
    # Test sizes - removed 121 (11x11, prime square causes issues)
    N_values = [15, 21, 33, 35, 39, 51, 55, 57, 65, 77, 85, 91, 95, 111, 115, 119, 133, 143, 155]
    
    print(f"\nAnalyzing {len(N_values)} problem sizes: {min(N_values)} to {max(N_values)}")
    print()
    
    results = []
    for N in N_values:
        print(f"Processing N={N}...", end=" ")
        result = analyze_N(N)
        results.append(result)
        print(f"✓ (Success: {result['success_rate']:.1f}%)")
    
    # Print results table
    print("\n" + "="*80)
    print(" "*25 + "RESULTS FOR PLOTTING")
    print("="*80)
    print("\nESSENTIAL METRICS TABLE:")
    print("-" * 80)
    print(f"{'N':<8} {'Factors':<12} {'Bases':<8} {'Success':<10} {'Fail':<8} {'Success%':<12} {'Avg Ops':<12} {'Time(ms)':<12}")
    print("-" * 80)
    
    for r in results:
        factors_str = 'x'.join(map(str, r['factors']))
        print(f"{r['N']:<8} {factors_str:<12} {r['num_bases']:<8} {r['success_count']:<10} {r['failure_count']:<8} "
              f"{r['success_rate']:<12.1f} {r['avg_operations']:<12.2f} {r['time_ms']:<12.3f}")
    
    print("-" * 80)
    
    # Summary statistics
    print("\n" + "="*80)
    print(" "*30 + "SUMMARY STATISTICS")
    print("="*80)
    
    avg_success_rate = np.mean([r['success_rate'] for r in results])
    total_time = sum([r['time_ms'] for r in results])
    
    print(f"\nOverall Performance:")
    print(f"  Average Success Rate:    {avg_success_rate:.1f}%")
    print(f"  Average Failure Rate:    {100 - avg_success_rate:.1f}%")
    print(f"  Total Computation Time:  {total_time:.3f} ms")
    print(f"  Avg Time per N:          {total_time/len(results):.3f} ms")
    
    # Scaling analysis
    first, last = results[0], results[-1]
    n_growth = last['N'] / first['N']
    ops_growth = last['avg_operations'] / first['avg_operations']
    time_growth = last['time_ms'] / first['time_ms']
    
    print(f"\nScaling Analysis (N={first['N']} → N={last['N']}):")
    print(f"  N increased by:         {n_growth:.2f}x")
    print(f"  Operations increased:   {ops_growth:.2f}x")
    print(f"  Time increased:         {time_growth:.2f}x")
    
    # Estimate complexity
    log_n_ratio = np.log(n_growth)
    log_ops_ratio = np.log(ops_growth)
    exponent = log_ops_ratio / log_n_ratio
    
    print(f"  Scaling complexity:     O(N^{exponent:.2f})")
    
    # Key metrics comparison table
    print("\n" + "="*80)
    print(" "*25 + "KEY ALGORITHM METRICS")
    print("="*80)
    
    print("\n┌─────────────────────────────────────┬──────────────────────────┐")
    print("│ Metric                              │ Value                    │")
    print("├─────────────────────────────────────┼──────────────────────────┤")
    print(f"│ Average Success Rate                │ {avg_success_rate:>22.1f}% │")
    print(f"│ Average Failure Rate                │ {100-avg_success_rate:>22.1f}% │")
    print(f"│ Deterministic per base?             │ {'Yes':>24} │")
    print(f"│ Fastest (N=15)                      │ {results[0]['time_ms']:>20.3f} ms │")
    print(f"│ Slowest (N={last['N']})                      │ {results[-1]['time_ms']:>20.3f} ms │")
    print(f"│ Computational Complexity            │ {f'O(N^{exponent:.2f})':>24} │")
    print(f"│ Scales well for large N?            │ {'No (polynomial)':>24} │")
    print("└─────────────────────────────────────┴──────────────────────────┘")
    
    # Data arrays for plotting
    print("\n" + "="*80)
    print(" "*25 + "DATA FOR YOUR PLOTS")
    print("="*80)
    
    print("\n📊 PLOT 1: Operations Scaling (Operations vs N)")
    print("-" * 80)
    print("N =", [r['N'] for r in results])
    print("Ops =", [round(r['avg_operations'], 2) for r in results])
    
    print("\n📊 PLOT 2: Success Rate vs N")
    print("-" * 80)
    print("N =", [r['N'] for r in results])
    print("Success% =", [round(r['success_rate'], 1) for r in results])
    
    print("\n📊 PLOT 3: Execution Time vs N")
    print("-" * 80)
    print("N =", [r['N'] for r in results])
    print("Time(ms) =", [round(r['time_ms'], 3) for r in results])
    
    print("\n📊 PLOT 4: Failure Rate vs N")
    print("-" * 80)
    print("N =", [r['N'] for r in results])
    print("Failure% =", [round(100 - r['success_rate'], 1) for r in results])
    
    # Detailed analysis of specific cases
    print("\n" + "="*80)
    print(" "*25 + "DETAILED CASE ANALYSIS")
    print("="*80)
    
    print("\nN=15 (3×5) Analysis:")
    r15 = results[0]
    print(f"  Bases tested: {r15['num_bases']}")
    print(f"  Successful: {r15['success_count']} bases ({r15['success_rate']:.1f}%)")
    print(f"  Failed: {r15['failure_count']} bases")
    print(f"  Avg operations: {r15['avg_operations']:.2f}")
    print(f"  Time: {r15['time_ms']:.3f} ms")
    
    print("\nN=21 (3×7) Analysis:")
    r21 = results[1]
    print(f"  Bases tested: {r21['num_bases']}")
    print(f"  Successful: {r21['success_count']} bases ({r21['success_rate']:.1f}%)")
    print(f"  Failed: {r21['failure_count']} bases")
    print(f"  Avg operations: {r21['avg_operations']:.2f}")
    print(f"  Time: {r21['time_ms']:.3f} ms")
    
    # Check for any anomalies
    print("\n" + "="*80)
    print(" "*25 + "DATA VALIDATION")
    print("="*80)
    
    print("\nChecking for anomalies:")
    issues = []
    for r in results:
        if r['success_rate'] == 0:
            issues.append(f"  ⚠ N={r['N']} has 0% success rate (factors: {r['factors']})")
        if r['success_rate'] < 20:
            issues.append(f"  ⚠ N={r['N']} has very low success rate: {r['success_rate']:.1f}%")
    
    if issues:
        print("\nPotential issues found:")
        for issue in issues:
            print(issue)
    else:
        print("  ✓ All data looks valid")
        print("  ✓ Success rates are reasonable")
        print("  ✓ Timing data excludes initialization overhead")
    
    print("\n" + "="*80)
    print(" "*30 + "ANALYSIS COMPLETE")
    print("="*80)
    print("\nCopy the arrays above directly into your plotting tool!")
    print("="*80 + "\n")

if __name__ == "__main__":
    main()


                    CLASSICAL SHOR'S ALGORITHM
                         ESSENTIAL METRICS

Analyzing 19 problem sizes: 15 to 155

Processing N=15... ✓ (Success: 85.7%)
Processing N=21... ✓ (Success: 54.5%)
Processing N=33... ✓ (Success: 52.6%)
Processing N=35... ✓ (Success: 78.3%)
Processing N=39... ✓ (Success: 78.3%)
Processing N=51... ✓ (Success: 96.8%)
Processing N=55... ✓ (Success: 76.9%)
Processing N=57... ✓ (Success: 51.4%)
Processing N=65... ✓ (Success: 63.8%)
Processing N=77... ✓ (Success: 50.8%)
Processing N=85... ✓ (Success: 92.1%)
Processing N=91... ✓ (Success: 76.1%)
Processing N=95... ✓ (Success: 76.1%)
Processing N=111... ✓ (Success: 76.1%)
Processing N=115... ✓ (Success: 75.9%)
Processing N=119... ✓ (Success: 94.7%)
Processing N=133... ✓ (Success: 50.5%)
Processing N=143... ✓ (Success: 75.6%)
Processing N=155... ✓ (Success: 75.6%)

                         RESULTS FOR PLOTTING

ESSENTIAL METRICS TABLE:
--------------------------------------------------------------------